In [5]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

import folium
from folium.plugins import HeatMap, HeatMapWithTime
from pandas.api.types import CategoricalDtype

In [3]:
df = pd.read_csv("../data/trees-with-species-and-dimensions-urban-forest.csv") # 19 MB
df = df.drop(['Precinct', 'geolocation'], axis=1)
# df = df[df['Year Planted'] > 2004]
# df['Year Planted'] = pd.to_datetime(df['Year Planted'], format='%Y')
df.head()

,CoM ID,Common Name,Scientific Name,Genus,Family,Diameter Breast Height,Year Planted,Date Planted,Age Description,Useful Life Expectency,Useful Life Expectency Value,Located in,UploadDate,CoordinateLocation,Latitude,Longitude,Easting,Northing
0,1440992,River red gum,Eucalyptus camaldulensis,Eucalyptus,Myrtaceae,NaN,2009,2009-12-14,NaN,NaN,NaN,Park,2021-01-10,"-37.789042536009, 144.94750113149306",-37.789043,144.947501,319271.37,5815606.69
1,1286119,River red gum,Eucalyptus camaldulensis,Eucalyptus,Myrtaceae,80.0,2008,2008-07-16,Mature,31-60 years,60.0,Park,2021-01-10,"-37.78989006812276, 144.9256959906416",-37.789890,144.925696,317353.24,5815470.25
2,1439848,European nettle tree,Celtis australis,Celtis,Cannabaceae,4.0,2009,2009-09-08,Juvenile,31-60 years,60.0,Street,2021-01-10,"-37.795227592098875, 144.91940533967247",-37.795228,144.919405,316812.46,5814865.65
3,1584631,Swamp Sheoak,Casuarina obesa,Casuarina,Casuarinaceae,NaN,2015,2015-06-18,NaN,NaN,NaN,Park,2021-01-10,"-37.795178798251044, 144.95235531785673",-37.795179,144.952355,319713.76,5814935.15
4,1286271,Golden Poplar,Populus x canadensis,Populus,Salicaceae,35.0,2008,2008-12-18,Semi-Mature,31-60 years,60.0,Street,2021-01-10,"-37.7904175404039, 144.92779056976474",-37.790418,144.927791,317538.99,5815415.81


In [ ]:

# markers = np.array([
#     [37.80029, -122.41018, 'North Beach'],
#     [37.79417, -122.40694, 'Chinatown'],
#     [37.77722, -122.41111, 'South of Market'],
#     [37.78806, -122.4075, 'Union Square'],
#     [37.7825, -122.4108, 'Theatre District'],
#     [37.76083, -122.435, 'Castro District'],
#     [37.803, -122.436, 'Marina District'],
#     [37.791190, -122.420828, 'Polk Street'],
#     [37.76, -122.42, 'Mission'],
# ])

# polk_street = [(37.806217, -122.423863), (37.776601, -122.417903)]

In [18]:
# data_party_geo = data_party[['Y', 'X']]
map_center = [-37.810935, 144.946457]  # Melbourne coords
# tree_map = folium.Map(location=map_center, zoom_start=12, width=900, height=500)

In [23]:
# data_party_geo = data_party.loc[(data_party['Hour'] > 17) | (data_party['Hour'] < 6)]
import branca.colormap as cm

years = ['2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021']
cat_type = CategoricalDtype(categories=years, ordered=True)
df['YearC'] = df['Year Planted'].astype(str).astype(cat_type)

df_geo = [[[row['Latitude'], row['Longitude']] for _, row in df[df['YearC'] == h].iterrows()] for h in years]

tree_map = folium.Map(location=map_center, zoom_start=13, width=900, height=500) # wysokosc i zoom!

# SF_map_play = folium.Map(location=[37.7749, -122.4194], zoom_start=13)
# for lat, lon, name in markers:
#     folium.Marker([lat, lon], popup=name).add_to(SF_map_play)

# folium.PolyLine(polk_street, weight=4).add_to(SF_map_play)
# gradient = {.33: 'red', .66: 'brown', 1: 'green'}
HeatMapWithTime(df_geo, gradient=cm.linear.PuBuGn_09.to_step(12), auto_play=True, index=years).add_to(tree_map)
tree_map

In [30]:
# cm.linear.PuBuGn_09.to_step(12)

In [83]:
import geopandas as gpd
import pandas as pd
import json

districts_gdp = gpd.read_file('../data/small-areas-for-census-of-land-use-and-employment-clue.geojson')
districts_gdf = gpd.GeoDataFrame(districts_gdp)

trees = pd.read_csv("../data/trees-with-species-and-dimensions-urban-forest.csv").drop('Precinct', axis=1)

trees_gdf = gpd.GeoDataFrame(trees, geometry=gpd.points_from_xy(trees.Longitude, trees.Latitude))

# Perform spatial join
trees_with_districts = gpd.sjoin(trees_gdf, districts_gdf, how="left", predicate="within")

trees_with_districts = trees_with_districts.drop(
    ['geometry', 'geo_point_2d', 'index_right', 'geolocation'], axis=1)

C:\Users\agama\AppData\Local\Temp\ipykernel_23708\2143501429.py:13: UserWarning:

CRS mismatch between the CRS of left geometries and the CRS of right geometries.
Use `to_crs()` to reproject one of the input geometries to match the CRS of the other.

Left CRS: None
Right CRS: EPSG:4326




In [84]:
districts_gdp = districts_gdp.rename(columns={'featurenam':'District'})
districts_gdp.index = districts_gdp['District']

#Dropping useless columns for this application (it's all about saving memory)
districts_gdp.drop(['District', 'geo_point_2d', 'shape_area', 'shape_len'], axis=1, inplace=True)

#Choropleth mapbox accepts a json for the geometries of neighbourhoods.
districts_json = json.loads(districts_gdp.to_json())
districts_gdp.head()

,geometry
District,
Kensington,"MULTIPOLYGON (((144.93687 -37.78884, 144.93667..."
West Melbourne (Residential),"MULTIPOLYGON (((144.95144 -37.81317, 144.95033..."
Melbourne (CBD),"MULTIPOLYGON (((144.95144 -37.81317, 144.95392..."
Melbourne (Remainder),"MULTIPOLYGON (((144.98502 -37.84568, 144.98420..."
Parkville,"MULTIPOLYGON (((144.94037 -37.78762, 144.94007..."


In [85]:
trees_with_districts = trees_with_districts.rename(columns={'featurenam':'District'})
df = trees_with_districts.groupby(['District', 'Year Planted'])['CoM ID'].count().reset_index().sort_values('Year Planted')
df['Year Planted'] = df['Year Planted'].astype(str)
df = df[df['Year Planted'] >= '2012']
df = df.rename(columns={'CoM ID': 'Number of planted trees'})

In [88]:
#using plotly for an animated choropleth map
import plotly.express as px
fig = px.choropleth_mapbox(data_frame=df,
                           geojson=districts_json,
                           locations=df.District,
                           color='Number of planted trees',
                           center={'lat': map_center[0], 'lon': map_center[1]},
                           mapbox_style='open-street-map',
                           zoom=11,
                           color_continuous_scale='greens',
                           range_color=(0, 1500),
                           opacity=0.7,
                           animation_frame='Year Planted',
                           width=800,
                           height=600)
fig.write_html('../plots/district_interactive_map.html')
fig.show()